# Notebook 3 — RAG Chain

This notebook builds the core question-answering pipeline. It takes a natural
language query, retrieves the most relevant chunks from ChromaDB, and passes
them to GPT-4o-mini with a carefully designed prompt that instructs the model
to answer strictly from the retrieved context and cite its sources. This is the
pipeline that will later be wrapped in a FastAPI endpoint.

In [1]:
import sys
from dotenv import load_dotenv
import os

load_dotenv()

print(sys.executable)
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith key loaded:", bool(os.getenv("LANGSMITH_API_KEY")))

c:\Users\nabee\anaconda3\envs\rag3gpp\python.exe
OpenAI key loaded: True
LangSmith key loaded: True


## LangSmith tracing setup

LangSmith intercepts every LangChain call and logs the full trace — which
chunks were retrieved, what prompt was sent to the LLM, and what it returned.
This is how we debug retrieval failures and monitor answer quality in
production. Setting these environment variables is all that's needed to
activate it.

In [2]:
import langsmith

# activate LangSmith tracing for this session
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "3gpp-rag"

# confirm the LangSmith client can connect
client = langsmith.Client()
print("LangSmith connected:", client.api_key is not None)

LangSmith connected: True


## Prompt template

The prompt explicitly instructs GPT-4o-mini to answer only from the provided
context and to cite the source document and page number for every claim. The
"I don't know" instruction is critical — it prevents the model from filling
gaps with hallucinated technical details, which would be dangerous in a
standards reference tool.

In [3]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template("""
You are a technical assistant specializing in 3GPP wireless standards.
Answer the question using ONLY the context provided below.
For every piece of information you use, cite the source document and page number.
If the answer is not in the context, say "I don't have enough information in the
loaded specifications to answer this question."

Context:
{context}

Question:
{question}

Answer (with citations):
""")

print("Prompt template created successfully")
print(f"\nInput variables: {prompt_template.input_variables}")

Prompt template created successfully

Input variables: ['context', 'question']


## Building the RAG chain

We connect ChromaDB, the prompt template, and GPT-4o-mini into a single
pipeline using LangChain's LCEL (LangChain Expression Language) syntax.
The chain retrieves the top 5 most relevant chunks, formats them with their
source metadata, fills the prompt template, and passes it to the LLM.
Every run is automatically traced in LangSmith.

In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# load the vectorstore from disk
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma(
    persist_directory=r"C:\Projects\3gpp-rag\chroma_db",
    embedding_function=embeddings
)

# set up the retriever - k=5 returns the 5 most relevant chunks
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# set up the LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# helper function to format retrieved chunks with their source info
def format_docs(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get("title", "Unknown")
        page = doc.metadata.get("page", "Unknown")
        formatted.append(f"[{source}, Page {page}]\n{doc.page_content}")
    return "\n\n".join(formatted)

# build the chain using LCEL pipe syntax
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully")

RAG chain built successfully


## Running the chain

Testing the full pipeline end to end with a technical question about NR
physical channels. A good answer will reference specific clauses or tables
from the specs and cite the source document and page number.

In [5]:
question = "What are the supported modulation orders Qm for PDSCH including QPSK 16QAM 64QAM 256QAM?"

answer = rag_chain.invoke(question)

print(f"Question: {question}")
print(f"\nAnswer:\n{answer}")

Question: What are the supported modulation orders Qm for PDSCH including QPSK 16QAM 64QAM 256QAM?

Answer:
The supported modulation orders Qm for PDSCH are as follows:

- QPSK: 2 (3GPP TS 38.211, Page 32)
- 16QAM: 4 (3GPP TS 38.211, Page 32)
- 64QAM: 6 (3GPP TS 38.211, Page 32)
- 256QAM: 8 (3GPP TS 38.211, Page 32)
